# Comprehensive PGNN Pipeline — Al 6011-O Flow-Stress Prediction

**Models trained:**

| # | Name | Description |
|---|------|-------------|
| 1 | ANN  | Black-box MLP baseline (no physics) |
| 2 | PGNN | Arrhenius architecture, data loss only |
| 3 | PGNN + λA | Warmup → grid-search λ |
| 4 | PGNN + λB | Adaptive loss-ratio λ |
| 5 | PGNN + λC | Gradient-norm balancing λ |

**Analysis sections:**
- Per-condition R², RMSE, AARE for all models
- Physical insight: learned Arrhenius parameters vs SCAM & literature
- Activation energy analysis & parameter–feature correlations

Physics anchor: strain-compensated Arrhenius model (SCAM) polynomial.

## 1 · Imports & Configuration

In [ ]:
from __future__ import annotations

import copy
import os
import random
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


# ── Hyperparameters & paths in one place ──────────────────────────
@dataclass
class Config:
    """Central configuration — edit here, nowhere else."""

    # Paths
    data_path: str = "/kaggle/input/datasets/vatine/al6011data/al6011_downsampled_full.xlsx"
    output_dir: str = "."

    # Reproducibility
    seed: int = 42

    # Data
    feature_cols: List[str] = field(default_factory=lambda: ["T_K", "ln_sr", "eps_true"])
    target_col: str = "sigma_true"
    train_ratio: float = 0.70
    val_ratio: float = 0.15

    # Model
    hidden_dims: List[int] = field(default_factory=lambda: [128, 128, 64])
    dropout: float = 0.1

    # Training
    batch_size: int = 64
    epochs: int = 500
    patience: int = 60
    lr: float = 1e-3
    weight_decay: float = 1e-5
    scheduler_factor: float = 0.5
    scheduler_patience: int = 30
    grad_clip: float = 10.0

    # λ strategies
    lambda_grid_coarse: List[float] = field(default_factory=lambda: [0.01, 0.1, 0.5, 1.0])
    lambda_grid_fine: List[float] = field(
        default_factory=lambda: [0.001, 0.003, 0.005, 0.007, 0.01, 0.015, 0.02, 0.03, 0.05]
    )
    warmup_frac_A: float = 0.20          # Strategy A
    adaptive_lr_B: float = 0.10          # Strategy B
    ema_beta_C: float = 0.90             # Strategy C
    warmup_frac_C: float = 0.10          # Strategy C

    # MC Dropout
    mc_passes: int = 100

    # Physics mask threshold (250 °C ≈ 523.15 K)
    scam_temp_min_K: float = 523.0

    @property
    def device(self) -> torch.device:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG = Config()
print(f"Device : {CFG.device}")
print(f"Epochs : {CFG.epochs}  |  Patience : {CFG.patience}")
print(f"Hidden : {CFG.hidden_dims}  |  Dropout : {CFG.dropout}")

## 2 · Reproducibility

In [ ]:
def seed_everything(seed: int = CFG.seed) -> None:
    """Lock all RNG sources for reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()

## 3 · SCAM Polynomial (Physics Anchor)

Strain-compensated Arrhenius model fitted to 250–450 °C data.
Provides α(ε), n(ε), Q(ε), ln A(ε) as 6th-degree polynomials.

In [ ]:
R_GAS = 8.314  # J/(mol·K)

SCAM_POLY_COEFFS: Dict[str, np.ndarray] = {
    "alpha": np.array([ 2.990838465670e+04, -1.994069281431e+04,  5.345762450621e+03,
                        -7.350303313298e+02,  5.448368675032e+01, -2.063309588944e+00,
                         5.051186723984e-02]),
    "n":     np.array([-7.812215710077e+05,  2.343228580944e+05,  4.087839767037e+04,
                        -2.591299218747e+04,  4.249708974788e+03, -3.168428320818e+02,
                         1.508120816407e+01]),
    "Q":     np.array([-8.543584880384e+10,  5.238248618259e+10, -1.249643528684e+10,
                         1.436699072794e+09, -7.644242300049e+07,  1.055736379502e+06,
                         1.900591723054e+05]),
    "lnA":   np.array([-2.689844217277e+07,  1.687345535442e+07, -4.163261535534e+06,
                         5.066408364188e+05, -3.055325125124e+04,  7.305787733034e+02,
                         2.523144522408e+01]),
}

# Torch copies — initialised lazily on the training device
_SCAM_TORCH: Dict[str, torch.Tensor] = {}


def init_scam_on_device(dev: torch.device) -> None:
    """Move SCAM coefficients to *dev* as float32 tensors."""
    global _SCAM_TORCH
    _SCAM_TORCH = {
        k: torch.tensor(v, dtype=torch.float32, device=dev)
        for k, v in SCAM_POLY_COEFFS.items()
    }


def _polyval_torch(coeffs: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Horner's method — fully differentiable."""
    out = coeffs[0] * torch.ones_like(x)
    for c in coeffs[1:]:
        out = out * x + c
    return out


def scam_sigma_np(eps: np.ndarray, T_K: np.ndarray, sr: np.ndarray) -> np.ndarray:
    """Compute σ_SCAM in NumPy (for precomputing targets)."""
    a   = np.polyval(SCAM_POLY_COEFFS["alpha"], eps)
    n   = np.polyval(SCAM_POLY_COEFFS["n"],     eps)
    Q   = np.polyval(SCAM_POLY_COEFFS["Q"],     eps)
    lnA = np.polyval(SCAM_POLY_COEFFS["lnA"],   eps)
    Z = sr * np.exp(Q / (R_GAS * T_K))
    return (1.0 / a) * np.arcsinh((Z / np.exp(lnA)) ** (1.0 / n))

## 4 · Data Loading & Stratified Split

In [ ]:
# ── Load all condition sheets ─────────────────────────────────────
xf = pd.ExcelFile(CFG.data_path)
sheets = [s for s in xf.sheet_names if s != "Summary"]

data = pd.concat(
    [pd.read_excel(CFG.data_path, sheet_name=s).assign(condition=s) for s in sheets],
    ignore_index=True,
)
print(f"Loaded {len(data):,} points  |  {data['condition'].nunique()} conditions")

# ── Feature / target arrays ───────────────────────────────────────
X_all = data[CFG.feature_cols].values.astype(np.float32)
y_all = data[CFG.target_col].values.astype(np.float32).reshape(-1, 1)

# Raw (unscaled) columns needed by the Arrhenius layer & SCAM
T_K_raw = data["T_K"].values.astype(np.float32)
sr_raw  = data["strain_rate"].values.astype(np.float32)
eps_raw = data["eps_true"].values.astype(np.float32)

# ── Stratified train / val / test split ───────────────────────────
seed_everything()

train_idx, val_idx, test_idx = [], [], []
for cond in data["condition"].unique():
    idx = data[data["condition"] == cond].index.values.copy()
    np.random.shuffle(idx)
    n = len(idx)
    n_tr = int(CFG.train_ratio * n)
    n_va = int(CFG.val_ratio * n)
    train_idx.extend(idx[:n_tr])
    val_idx.extend(idx[n_tr : n_tr + n_va])
    test_idx.extend(idx[n_tr + n_va :])

print(f"Split  : {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test")

## 5 · Feature Scaling & SCAM Pre-computation

In [ ]:
# ── StandardScaler (fit on train only) ────────────────────────────
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_s = scaler_X.fit_transform(X_all[train_idx]).astype(np.float32)
y_train_s = scaler_y.fit_transform(y_all[train_idx]).astype(np.float32)

# ── Precompute σ_SCAM for every point ────────────────────────────
sigma_scam_raw = scam_sigma_np(eps_raw, T_K_raw, sr_raw).astype(np.float64)

# Physics mask: valid only where SCAM polynomials were fitted (250–450 °C)
physics_valid = np.isfinite(sigma_scam_raw) & (T_K_raw >= CFG.scam_temp_min_K)
sigma_scam_raw[~physics_valid] = 0.0
sigma_scam_raw = np.clip(sigma_scam_raw, 0, 500).astype(np.float32).reshape(-1, 1)

sigma_scam_scaled = scaler_y.transform(sigma_scam_raw).astype(np.float32)
physics_mask      = physics_valid.astype(np.float32).reshape(-1, 1)

print(f"SCAM valid : {physics_valid.sum()}/{len(physics_valid)} points")
print(f"σ_SCAM range (valid): [{sigma_scam_raw[physics_valid].min():.1f}, "
      f"{sigma_scam_raw[physics_valid].max():.1f}] MPa")

## 6 · DataLoaders

In [ ]:
def make_loader(idx_list: List[int], shuffle: bool) -> DataLoader:
    """Build a DataLoader carrying scaled features, raw physics inputs,
    scaled SCAM targets, and the physics mask."""
    tensors = (
        torch.tensor(scaler_X.transform(X_all[idx_list]).astype(np.float32)),
        torch.tensor(scaler_y.transform(y_all[idx_list]).astype(np.float32)),
        torch.tensor(T_K_raw[idx_list].reshape(-1, 1)),
        torch.tensor(sr_raw[idx_list].reshape(-1, 1)),
        torch.tensor(eps_raw[idx_list].reshape(-1, 1)),
        torch.tensor(sigma_scam_scaled[idx_list]),
        torch.tensor(physics_mask[idx_list]),
    )
    ds = TensorDataset(*tensors)
    bs = CFG.batch_size if shuffle else 256
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)


train_loader = make_loader(train_idx, shuffle=True)
val_loader   = make_loader(val_idx,   shuffle=False)
test_loader  = make_loader(test_idx,  shuffle=False)

print(f"Batches : {len(train_loader)} train / {len(val_loader)} val / {len(test_loader)} test")

## 7 · Model Architecture

**HybridPGNN**: shared MLP backbone → 4 sigmoid-bounded parameter heads
(α, n, Q, ln A) → differentiable Arrhenius formula → σ_pred.

The Arrhenius layer operates in *physical units*; scaling to match
training targets happens outside `forward`.

In [ ]:
# Physical-parameter bounds: (offset, range) so that
#   param = offset + range * sigmoid(logit)
PARAM_BOUNDS = {
    "alpha": (0.005,   0.045),    # MPa⁻¹
    "n":     (2.0,     8.0),
    "Q":     (130_000, 90_000),   # J/mol
    "lnA":   (15.0,    25.0),
}


class HybridPGNN(nn.Module):
    """Physics-Guided Neural Network with an Arrhenius output layer."""

    def __init__(
        self,
        input_dim: int = 3,
        hidden_dims: Optional[List[int]] = None,
        dropout: float = CFG.dropout,
    ) -> None:
        super().__init__()
        if hidden_dims is None:
            hidden_dims = CFG.hidden_dims

        # Shared MLP backbone
        layers: List[nn.Module] = []
        in_d = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        self.backbone = nn.Sequential(*layers)

        # One head per Arrhenius parameter
        self.heads = nn.ModuleDict({
            name: nn.Linear(hidden_dims[-1], 1) for name in PARAM_BOUNDS
        })

    # ──────────────────────────────────────────────────────────────
    def forward(
        self,
        x_scaled: torch.Tensor,
        T_K: torch.Tensor,
        sr: torch.Tensor,
        eps: torch.Tensor,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Parameters
        ----------
        x_scaled : (B, 3) scaled features [T_K, ln_sr, eps]
        T_K, sr, eps : (B, 1) raw physical values

        Returns
        -------
        sigma_pred : (B, 1)  predicted stress in MPa (unscaled)
        params     : dict of (B, 1) learned Arrhenius parameters
        """
        h = self.backbone(x_scaled)

        params: Dict[str, torch.Tensor] = {}
        for name, (lo, rng) in PARAM_BOUNDS.items():
            params[name] = lo + rng * torch.sigmoid(self.heads[name](h))

        # Arrhenius equation (with numerical safeguards)
        Z_exp  = torch.clamp(params["Q"] / (R_GAS * T_K), max=80.0)
        Z      = sr * torch.exp(Z_exp)
        ratio  = torch.clamp(Z / torch.exp(params["lnA"]), min=1e-10, max=1e30)
        sigma  = (1.0 / params["alpha"]) * torch.arcsinh(ratio ** (1.0 / params["n"]))

        return sigma, params

## 8 · Training Engine

A single `train_pgnn` function handles all four variants via the
`strategy` argument (`"none"`, `"A"`, `"B"`, `"C"`).

In [ ]:
@dataclass
class TrainHistory:
    """Lightweight container for training curves."""
    train_loss: List[float] = field(default_factory=list)
    val_loss:   List[float] = field(default_factory=list)
    data_loss:  List[float] = field(default_factory=list)
    phys_loss:  List[float] = field(default_factory=list)
    lam_hist:   List[float] = field(default_factory=list)
    best_epoch: int = 0


def _scale_sigma(raw: torch.Tensor) -> torch.Tensor:
    """Convert raw σ (MPa) to scaled space using scaler_y."""
    return (raw - scaler_y.mean_[0]) / scaler_y.scale_[0]


def _compute_physics_loss(
    pred_scaled: torch.Tensor,
    scam_scaled: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """Masked MSE between prediction and SCAM target (scaled space)."""
    n_valid = mask.sum()
    if n_valid == 0:
        return torch.tensor(0.0, device=pred_scaled.device)
    return ((pred_scaled - scam_scaled) ** 2 * mask).sum() / n_valid


def _grad_norm(model: nn.Module) -> float:
    """L2 norm of all parameter gradients (scalar)."""
    return sum(
        p.grad.data.norm(2).item() ** 2
        for p in model.parameters() if p.grad is not None
    ) ** 0.5


# ──────────────────────────────────────────────────────────────────
def train_pgnn(
    model: HybridPGNN,
    strategy: str = "none",
    lambda_max: float = 0.1,
    run_name: str = "PGNN",
    verbose_every: int = 50,
) -> Tuple[HybridPGNN, TrainHistory]:
    """
    Train a HybridPGNN model.

    Parameters
    ----------
    strategy   : "none" | "A" | "B" | "C"
    lambda_max : peak λ for Strategy A
    run_name   : label for log lines
    """
    model = model.to(CFG.device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=CFG.scheduler_factor,
        patience=CFG.scheduler_patience, min_lr=1e-6,
    )
    criterion = nn.MSELoss()
    hist = TrainHistory()

    best_val = float("inf")
    best_state = None
    wait = 0

    # Strategy state
    warmup_A  = int(CFG.warmup_frac_A * CFG.epochs)
    warmup_C  = int(CFG.warmup_frac_C * CFG.epochs)
    lam       = 0.0
    ema_lam   = 0.0

    for epoch in range(CFG.epochs):

        # ── λ schedule (beginning of epoch) ──────────────────────
        if strategy == "A":
            lam = lambda_max * min(epoch / warmup_A, 1.0)
        elif strategy == "C":
            frac = min(epoch / warmup_C, 1.0) if epoch < warmup_C else 1.0
            lam = ema_lam * frac

        # ── Train one epoch ──────────────────────────────────────
        model.train()
        sum_data = sum_phys = sum_total = 0.0
        n_samples = 0
        gn_data_sq = gn_phys_sq = 0.0
        gn_batches = 0

        for X_b, y_b, T_b, sr_b, eps_b, scam_b, pmask_b in train_loader:
            X_b, y_b     = X_b.to(CFG.device), y_b.to(CFG.device)
            T_b, sr_b    = T_b.to(CFG.device), sr_b.to(CFG.device)
            eps_b        = eps_b.to(CFG.device)
            scam_b       = scam_b.to(CFG.device)
            pmask_b      = pmask_b.to(CFG.device)

            sigma_raw, _ = model(X_b, T_b, sr_b, eps_b)
            sigma_s      = _scale_sigma(sigma_raw)

            L_data    = criterion(sigma_s, y_b)
            L_physics = _compute_physics_loss(sigma_s, scam_b, pmask_b)

            # Strategy C: collect per-batch gradient norms
            if strategy == "C":
                opt.zero_grad(); L_data.backward(retain_graph=True)
                gn_d = _grad_norm(model)
                opt.zero_grad(); L_physics.backward(retain_graph=True)
                gn_p = _grad_norm(model)
                gn_data_sq += gn_d ** 2
                gn_phys_sq += gn_p ** 2
                gn_batches += 1
                opt.zero_grad()

            loss = L_data if strategy == "none" else L_data + lam * L_physics

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            opt.step()

            bs = len(X_b)
            sum_data  += L_data.item() * bs
            sum_phys  += L_physics.item() * bs
            sum_total += loss.item() * bs
            n_samples += bs

        avg_data  = sum_data  / n_samples
        avg_phys  = sum_phys  / n_samples
        avg_total = sum_total / n_samples

        # ── Post-epoch λ updates ─────────────────────────────────
        if strategy == "C" and gn_batches > 0:
            raw_lam = (gn_data_sq / gn_batches) ** 0.5 / ((gn_phys_sq / gn_batches) ** 0.5 + 1e-8)
            ema_lam = CFG.ema_beta_C * ema_lam + (1 - CFG.ema_beta_C) * raw_lam
            lam = ema_lam
        if strategy == "B":
            if epoch == 0:
                lam = 0.1
            r = avg_data / (lam * avg_phys + 1e-8)
            lam *= (1 + CFG.adaptive_lr_B) if r > 1.2 else (1 - CFG.adaptive_lr_B) if r < 0.8 else 1.0
            lam = float(np.clip(lam, 1e-4, 10.0))

        hist.train_loss.append(avg_total)
        hist.data_loss.append(avg_data)
        hist.phys_loss.append(avg_phys)
        hist.lam_hist.append(lam)

        # ── Validation ───────────────────────────────────────────
        model.eval()
        val_sum = 0.0
        with torch.no_grad():
            for X_b, y_b, T_b, sr_b, eps_b, *_ in val_loader:
                X_b, y_b = X_b.to(CFG.device), y_b.to(CFG.device)
                T_b      = T_b.to(CFG.device)
                sr_b     = sr_b.to(CFG.device)
                eps_b    = eps_b.to(CFG.device)
                s_raw, _ = model(X_b, T_b, sr_b, eps_b)
                val_sum += criterion(_scale_sigma(s_raw), y_b).item() * len(X_b)

        val_loss = val_sum / len(val_idx)
        hist.val_loss.append(val_loss)
        sched.step(val_loss)

        if val_loss < best_val:
            best_val, hist.best_epoch, wait = val_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if (epoch + 1) % verbose_every == 0 or epoch == 0:
            lr_now = opt.param_groups[0]["lr"]
            lam_s = f"λ={lam:.4f}" if strategy != "none" else "—"
            print(f"  [{run_name}] ep {epoch+1:4d}  "
                  f"total={avg_total:.6f}  data={avg_data:.6f}  phys={avg_phys:.6f}  "
                  f"val={val_loss:.6f}  {lam_s}  lr={lr_now:.1e}")

        if wait >= CFG.patience:
            print(f"  [{run_name}] Early stop ep {epoch+1}  (best: {hist.best_epoch+1})")
            break

    model.load_state_dict(best_state)
    model.eval()
    print(f"  [{run_name}] Best val = {best_val:.6f} @ ep {hist.best_epoch+1}")
    return model, hist

## 9 · Evaluation Helpers

In [ ]:
def predict(model: HybridPGNN, idx_list: List[int]) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
    """Return σ_pred (MPa, unscaled) and learned Arrhenius params for *idx_list*."""
    model.eval()
    X_s   = torch.tensor(scaler_X.transform(X_all[idx_list]).astype(np.float32), device=CFG.device)
    T_t   = torch.tensor(T_K_raw[idx_list].reshape(-1, 1), device=CFG.device)
    sr_t  = torch.tensor(sr_raw[idx_list].reshape(-1, 1),  device=CFG.device)
    eps_t = torch.tensor(eps_raw[idx_list].reshape(-1, 1), device=CFG.device)
    with torch.no_grad():
        sigma, params = model(X_s, T_t, sr_t, eps_t)
    return sigma.cpu().numpy(), {k: v.cpu().numpy() for k, v in params.items()}


def aare(y_true: np.ndarray, y_pred: np.ndarray, floor: float = 5.0) -> float:
    """Average Absolute Relative Error (%), ignoring points below *floor*."""
    m = y_true.flatten() > floor
    if m.sum() == 0:
        return float("nan")
    return float(np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100)


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute R², RMSE, AARE for a single split."""
    return {
        "r2":   float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "aare": aare(y_true, y_pred),
    }


def evaluate(model: HybridPGNN, label: str = "") -> Dict[str, Dict[str, float]]:
    """Print & return metrics on train / val / test."""
    print(f"\n{'─'*60}\n  {label}\n{'─'*60}")
    results = {}
    for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        y_pred, _ = predict(model, idx)
        m = metrics(y_all[idx], y_pred)
        results[name] = m
        print(f"  {name:<6}  R²={m['r2']:.6f}  RMSE={m['rmse']:.4f}  AARE={m['aare']:.2f}%")
    return results


def compare_params(model: HybridPGNN, idx_list: List[int], label: str = "") -> None:
    """Compare learned Arrhenius params to SCAM polynomial (250–450 °C only)."""
    _, params = predict(model, idx_list)
    mask = T_K_raw[idx_list] >= CFG.scam_temp_min_K
    if mask.sum() == 0:
        print("  No valid 250–450 °C points for comparison.")
        return
    eps_v = eps_raw[idx_list][mask]
    scam = {k: np.polyval(SCAM_POLY_COEFFS[k], eps_v) for k in SCAM_POLY_COEFFS}
    print(f"\n  Learned vs SCAM ({mask.sum()} pts, 250–450 °C):")
    print(f"  {'Param':<6} {'Learned':>22}  {'SCAM':>22}")
    for pname, unit, divisor in [("alpha","MPa⁻¹",1), ("n","",1), ("Q","kJ/mol",1e3), ("lnA","",1)]:
        lv = params[pname][mask] / divisor
        sv = scam[pname] / divisor
        print(f"  {pname:<6} {lv.mean():.4f}±{lv.std():.4f} {unit:>8}  "
              f"{sv.mean():.4f}±{sv.std():.4f} {unit}")

## 10 · Train All PGNN Variants

In [ ]:
init_scam_on_device(CFG.device)

models:    Dict[str, HybridPGNN]  = {}
histories: Dict[str, TrainHistory] = {}
results:   Dict[str, Dict]        = {}

# ── Model 2: PGNN (data loss only) ───────────────────────────────
print("━" * 60, "\nModel 2 — PGNN (no physics loss)\n" + "━" * 60)
seed_everything()
m2 = HybridPGNN()
m2, h2 = train_pgnn(m2, strategy="none", run_name="PGNN")
models["PGNN"], histories["PGNN"] = m2, h2
results["PGNN"] = evaluate(m2, "PGNN (no physics loss)")
compare_params(m2, test_idx)

# ── Model 3: PGNN + λA (Warmup → Grid Search) ───────────────────
print("\n" + "━" * 60, "\nModel 3 — PGNN + λA (warmup + grid search)\n" + "━" * 60)

best_lam_a, best_val_r2, best_ma, best_ha = None, -1e9, None, None
for lam in CFG.lambda_grid_coarse:
    print(f"\n  λ_max = {lam}")
    seed_everything()
    ma = HybridPGNN()
    ma, ha = train_pgnn(ma, strategy="A", lambda_max=lam, run_name=f"λA({lam})")
    vr2 = r2_score(y_all[val_idx], predict(ma, val_idx)[0])
    print(f"    val R² = {vr2:.6f}")
    if vr2 > best_val_r2:
        best_val_r2, best_lam_a = vr2, lam
        best_ma, best_ha = copy.deepcopy(ma), ha

print(f"\n  Best coarse λ_max = {best_lam_a}  (val R² = {best_val_r2:.6f})")
models["PGNN+λA"], histories["PGNN+λA"] = best_ma, best_ha
results["PGNN+λA"] = evaluate(best_ma, f"PGNN + λA (λ_max={best_lam_a})")
compare_params(best_ma, test_idx)

# ── Model 4: PGNN + λB (Adaptive Ratio) ─────────────────────────
print("\n" + "━" * 60, "\nModel 4 — PGNN + λB (adaptive ratio)\n" + "━" * 60)
seed_everything()
m4 = HybridPGNN()
m4, h4 = train_pgnn(m4, strategy="B", run_name="PGNN-λB")
models["PGNN+λB"], histories["PGNN+λB"] = m4, h4
results["PGNN+λB"] = evaluate(m4, "PGNN + λB (adaptive ratio)")
compare_params(m4, test_idx)

# ── Model 5: PGNN + λC (Gradient Norm Balancing) ────────────────
print("\n" + "━" * 60, "\nModel 5 — PGNN + λC (gradient-norm balancing)\n" + "━" * 60)
seed_everything()
m5 = HybridPGNN()
m5, h5 = train_pgnn(m5, strategy="C", run_name="PGNN-λC")
models["PGNN+λC"], histories["PGNN+λC"] = m5, h5
results["PGNN+λC"] = evaluate(m5, "PGNN + λC (grad norm)")
compare_params(m5, test_idx)

# Save checkpoints
for tag, mdl in models.items():
    fname = f"best_{tag.replace('+','_').replace('λ','lam')}.pt"
    torch.save(mdl.state_dict(), fname)
    print(f"  Saved {fname}")

## 11 · Comparison Table

In [ ]:
print(f"\n{'Model':<16} {'Test R²':>9} {'RMSE':>10} {'AARE':>9}")
print("─" * 46)
for tag in ["PGNN", "PGNN+λA", "PGNN+λB", "PGNN+λC"]:
    t = results[tag]["test"]
    print(f"{tag:<16} {t['r2']:>9.6f} {t['rmse']:>9.4f} {t['aare']:>8.2f}%")

best_tag = max(results, key=lambda k: results[k]["test"]["r2"])
print(f"\nBest variant: {best_tag}")
best_model = models[best_tag]

## 12 · Visualisation

### 12a · Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (tag, h) in zip(axes.flat, histories.items()):
    ax.plot(h.train_loss, label="train (total)", alpha=0.8)
    ax.plot(h.val_loss,   label="val",           alpha=0.8)
    if tag != "PGNN":
        ax.plot(h.data_loss, label="data",    alpha=0.5, ls="--")
        ax.plot(h.phys_loss, label="physics", alpha=0.5, ls=":")
    ax.axvline(h.best_epoch, color="r", ls="--", alpha=0.4, label=f"best ({h.best_epoch+1})")
    ax.set(title=tag, xlabel="Epoch", ylabel="Loss (scaled MSE)", yscale="log")
    ax.legend(fontsize=7)
plt.suptitle("Training Curves — PGNN Variants", fontsize=14)
plt.tight_layout(); plt.savefig("pgnn_training_curves.png", dpi=150, bbox_inches="tight"); plt.show()

### 12b · λ Schedules

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tag in zip(axes, ["PGNN+λA", "PGNN+λB", "PGNN+λC"]):
    ax.plot(histories[tag].lam_hist, lw=1.5)
    ax.set(title=f"λ — {tag}", xlabel="Epoch", ylabel="λ", yscale="log")
plt.tight_layout(); plt.savefig("pgnn_lambda_history.png", dpi=150, bbox_inches="tight"); plt.show()

### 12c · Scatter Plots (Best Model)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, idx) in zip(axes, [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]):
    yp, _ = predict(best_model, idx)
    yt = y_all[idx]
    ax.scatter(yt, yp, alpha=0.5, s=15, edgecolors="none")
    hi = max(yt.max(), yp.max()) * 1.05
    ax.plot([0, hi], [0, hi], "r--", lw=1.5, label="y = x")
    ax.set(xlim=[0, hi], ylim=[0, hi], xlabel="Experimental σ (MPa)",
           ylabel="Predicted σ (MPa)", title=name, aspect="equal")
    ax.legend()
plt.suptitle(f"Scatter — {best_tag}", fontsize=14)
plt.tight_layout(); plt.savefig("pgnn_scatter.png", dpi=150, bbox_inches="tight"); plt.show()

### 12d · Flow-Stress Curves

In [ ]:
y_all_pred, all_params = predict(best_model, list(range(len(data))))
data["sigma_pred"] = y_all_pred.flatten()

temperatures = sorted(data["T_C"].unique())
strain_rates = sorted(data["strain_rate"].unique())
MARKERS = ["o", "s", "^"]
COLORS  = ["#e41a1c", "#377eb8", "#4daf4a"]

ncols = 4
nrows = (len(temperatures) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))

for i, T in enumerate(temperatures):
    ax = axes.flat[i]
    for j, sr in enumerate(strain_rates):
        sub = data[(data["T_C"] == T) & (data["strain_rate"] == sr)].sort_values("eps_true")
        if sub.empty:
            continue
        ax.scatter(sub["eps_true"], sub["sigma_true"], marker=MARKERS[j],
                   color=COLORS[j], s=20, alpha=0.6, label=f"{sr} s⁻¹ (exp)")
        ax.plot(sub["eps_true"], sub["sigma_pred"], color=COLORS[j], lw=1.5)
    ax.set(title=f"T = {T} °C", xlabel="True Strain", ylabel="True Stress (MPa)")
    if i == 0:
        ax.legend(fontsize=7)

for k in range(len(temperatures), len(axes.flat)):
    fig.delaxes(axes.flat[k])

plt.suptitle(f"{best_tag} — Predicted vs Experimental", fontsize=14, y=1.01)
plt.tight_layout(); plt.savefig("pgnn_flow_curves.png", dpi=150, bbox_inches="tight"); plt.show()

### 12e · Learned Parameters vs SCAM

In [ ]:
eps_smooth = np.linspace(0.05, 0.27, 200)
scam_curves = {k: np.polyval(v, eps_smooth) for k, v in SCAM_POLY_COEFFS.items()}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
labels = {"alpha": "α (MPa⁻¹)", "n": "n", "Q": "Q (J/mol)", "lnA": "ln A"}

for ax, pname in zip(axes.flat, labels):
    sc = ax.scatter(eps_raw, all_params[pname].flat, c=T_K_raw, cmap="coolwarm", s=5, alpha=0.6)
    ax.plot(eps_smooth, scam_curves[pname], "k--", lw=1.5, alpha=0.7, label="SCAM polynomial")
    ax.set(xlabel="True Strain", ylabel=labels[pname], title=labels[pname])
    ax.legend(fontsize=8)
    plt.colorbar(sc, ax=ax, label="T (K)")

plt.suptitle(f"Learned Arrhenius Parameters — {best_tag}", fontsize=14)
plt.tight_layout(); plt.savefig("pgnn_learned_params.png", dpi=150, bbox_inches="tight"); plt.show()

## 13 · MC Dropout — Uncertainty Quantification

Enable dropout at inference and run multiple stochastic forward passes
to approximate the predictive distribution.

In [ ]:
def _enable_dropout(model: nn.Module) -> None:
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


def mc_dropout_predict(
    model: HybridPGNN, idx_list: List[int], T: int = CFG.mc_passes,
) -> np.ndarray:
    """Run *T* stochastic forward passes. Returns shape (T, N)."""
    model.eval()
    _enable_dropout(model)

    X_s   = torch.tensor(scaler_X.transform(X_all[idx_list]).astype(np.float32), device=CFG.device)
    T_t   = torch.tensor(T_K_raw[idx_list].reshape(-1, 1), device=CFG.device)
    sr_t  = torch.tensor(sr_raw[idx_list].reshape(-1, 1),  device=CFG.device)
    eps_t = torch.tensor(eps_raw[idx_list].reshape(-1, 1), device=CFG.device)

    preds = []
    with torch.no_grad():
        for _ in range(T):
            sigma, _ = model(X_s, T_t, sr_t, eps_t)
            preds.append(sigma.cpu().numpy().squeeze(-1))
    return np.array(preds)  # (T, N)


print(f"Running {CFG.mc_passes} MC passes on test set …")
mc = mc_dropout_predict(best_model, test_idx)

mc_mean = mc.mean(axis=0)
mc_std  = mc.std(axis=0)
ci_lo   = mc_mean - 1.96 * mc_std
ci_hi   = mc_mean + 1.96 * mc_std
y_test  = y_all[test_idx].flatten()

picp = float(((y_test >= ci_lo) & (y_test <= ci_hi)).mean() * 100)
mpiw = float((ci_hi - ci_lo).mean())
print(f"  PICP = {picp:.1f}%  (target ≥ 95%)")
print(f"  MPIW = {mpiw:.2f} MPa")
print(f"  Mean σ_unc = {mc_std.mean():.2f} MPa")

# ── Plot ──
order = np.argsort(y_test)
fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(range(len(order)), ci_lo[order], ci_hi[order],
                alpha=0.3, color="steelblue", label="95 % CI")
ax.plot(mc_mean[order], "b-", lw=0.8, label="MC mean")
ax.plot(y_test[order], "r.", ms=3, alpha=0.7, label="Experimental")
ax.set(xlabel="Test samples (sorted by σ)", ylabel="True Stress (MPa)",
       title=f"MC Dropout UQ — {best_tag}  |  PICP={picp:.1f}%  MPIW={mpiw:.2f} MPa")
ax.legend()
plt.tight_layout(); plt.savefig("pgnn_mc_dropout_uq.png", dpi=150, bbox_inches="tight"); plt.show()

## 14 · Per-Condition AARE Analysis

In [ ]:
# Predict with PGNN (no λ) and best λA model on all data
data["pred_noλ"] = predict(models["PGNN"], list(range(len(data))))[0].flatten()
data["pred_λA"]  = predict(models["PGNN+λA"], list(range(len(data))))[0].flatten()

data["split"] = "N/A"
for idx, lbl in [(train_idx, "train"), (val_idx, "val"), (test_idx, "test")]:
    data.loc[idx, "split"] = lbl


def _condition_aare(cond_df: pd.DataFrame, pred_col: str) -> float:
    yt = cond_df["sigma_true"].values
    yp = cond_df[pred_col].values
    return aare(yt, yp)


rows = []
for cond in data["condition"].unique():
    sub = data[data["condition"] == cond]
    T_C = sub["T_C"].iloc[0] if "T_C" in sub.columns else None
    sr = sub["strain_rate"].iloc[0]
    for sp in ["train", "val", "test"]:
        s = sub[sub["split"] == sp]
        if s.empty:
            continue
        a_noλ = _condition_aare(s, "pred_noλ")
        a_λA  = _condition_aare(s, "pred_λA")
        rows.append(dict(condition=cond, T_C=T_C, strain_rate=sr, split=sp,
                         n_pts=len(s), AARE_noλ=a_noλ, AARE_λA=a_λA,
                         delta=a_noλ - a_λA, has_physics=T_C is not None and T_C >= 250))

cond_df = pd.DataFrame(rows).sort_values(["T_C", "strain_rate", "split"])

# ── Print test-set table ─────────────────────────────────────────
test_cond = cond_df[cond_df["split"] == "test"]
print(f"{'Condition':<20} {'T(°C)':>6} {'SR':>10} {'Pts':>4} {'PGNN':>7} {'λA':>7} {'Δ':>7} {'Phys':>5}")
print("─" * 75)
for _, r in test_cond.iterrows():
    print(f"{r['condition']:<20} {r['T_C']:6.0f} {r['strain_rate']:10.4f} "
          f"{r['n_pts']:4} {r['AARE_noλ']:6.2f}% {r['AARE_λA']:6.2f}% "
          f"{r['delta']:+6.2f}% {'Y' if r['has_physics'] else 'N':>5}")

# ── Summary by temperature regime ────────────────────────────────
for sp in ["train", "val", "test"]:
    sd = cond_df[cond_df["split"] == sp]
    hi = sd[sd["has_physics"]]
    lo = sd[~sd["has_physics"]]
    print(f"\n  {sp.upper():5s}  250–450 °C: PGNN={hi['AARE_noλ'].mean():.2f}%  λA={hi['AARE_λA'].mean():.2f}%  Δ={hi['delta'].mean():+.2f}%"
          if len(hi) else "")
    if len(lo):
        print(f"         RT+150 °C: PGNN={lo['AARE_noλ'].mean():.2f}%  λA={lo['AARE_λA'].mean():.2f}%  Δ={lo['delta'].mean():+.2f}%")

# ── Heatmaps ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (label, col) in zip(axes, [("PGNN (no λ)", "AARE_noλ"), ("PGNN+λA", "AARE_λA")]):
    piv = test_cond.pivot_table(index="T_C", columns="strain_rate", values=col)
    im = ax.imshow(piv.values, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=20)
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f"{sr:.4f}" for sr in piv.columns], rotation=45)
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels([f"{T:.0f} °C" for T in piv.index])
    ax.set(xlabel="Strain Rate (s⁻¹)", ylabel="Temperature", title=f"{label} — Test AARE (%)")
    for i in range(len(piv.index)):
        for j in range(len(piv.columns)):
            v = piv.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                        fontsize=9, color="white" if v > 12 else "black")
    plt.colorbar(im, ax=ax, label="AARE (%)")

plt.suptitle("Per-Condition AARE Heatmap (Test)", fontsize=14)
plt.tight_layout(); plt.savefig("pgnn_aare_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

# ── Bar chart ─────────────────────────────────────────────────────
ts = test_cond.sort_values("AARE_λA", ascending=False)
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(ts)); w = 0.35
ax.bar(x - w/2, ts["AARE_noλ"], w, label="PGNN", color="#e41a1c", alpha=0.7)
ax.bar(x + w/2, ts["AARE_λA"],  w, label="PGNN+λA", color="#377eb8", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['T_C']:.0f}°C\n{r['strain_rate']:.4f}" for _, r in ts.iterrows()],
                   fontsize=7, rotation=45, ha="right")
ax.set(ylabel="AARE (%)", title="Per-Condition AARE — Test (worst → best)")
ax.legend()
plt.tight_layout(); plt.savefig("pgnn_aare_bar.png", dpi=150, bbox_inches="tight"); plt.show()

cond_df.to_csv("pgnn_percondition_aare.csv", index=False)
print("Saved: pgnn_percondition_aare.csv")

## 15 · Fine λ Grid Search (Strategy A)

In [ ]:
print("━" * 60, "\nFine λ Grid — Strategy A\n" + "━" * 60)

fine_results:   Dict[float, Dict] = {}
fine_models:    Dict[float, HybridPGNN] = {}
fine_histories: Dict[float, TrainHistory] = {}

for lam in CFG.lambda_grid_fine:
    print(f"\n  λ_max = {lam}")
    seed_everything()
    mf = HybridPGNN()
    mf, hf = train_pgnn(mf, strategy="A", lambda_max=lam, run_name=f"λA({lam})")

    res = {}
    for sp, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        yp, _ = predict(mf, idx)
        res[sp] = metrics(y_all[idx], yp)
    res["best_epoch"] = hf.best_epoch + 1

    fine_results[lam]   = res
    fine_models[lam]    = copy.deepcopy(mf)
    fine_histories[lam] = hf

    print(f"    train R²={res['train']['r2']:.6f}  val R²={res['val']['r2']:.6f}  "
          f"test R²={res['test']['r2']:.6f}  test AARE={res['test']['aare']:.2f}%")

# ── Table ─────────────────────────────────────────────────────────
print(f"\n{'λ':>8} {'Ep':>5} {'Tr R²':>9} {'Val R²':>9} {'Tst R²':>9} {'Tst RMSE':>10} {'Tst AARE':>10}")
print("─" * 65)
for lam in CFG.lambda_grid_fine:
    r = fine_results[lam]
    print(f"{lam:8.3f} {r['best_epoch']:5} "
          f"{r['train']['r2']:9.6f} {r['val']['r2']:9.6f} {r['test']['r2']:9.6f} "
          f"{r['test']['rmse']:10.4f} {r['test']['aare']:9.2f}%")

best_fine_lam = max(fine_results, key=lambda k: fine_results[k]["val"]["r2"])
print(f"\nBest fine λ_max = {best_fine_lam}  "
      f"(val R²={fine_results[best_fine_lam]['val']['r2']:.6f}  "
      f"test AARE={fine_results[best_fine_lam]['test']['aare']:.2f}%)")

torch.save(fine_models[best_fine_lam].state_dict(),
           f"best_pgnn_lam_A_fine_{best_fine_lam}.pt")

# ── Metric curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
lams = list(fine_results)
for ax, met, ylabel in zip(axes, ["r2", "rmse", "aare"], ["R²", "RMSE (MPa)", "AARE (%)"]):
    for sp, m in [("train", "o-"), ("val", "s-"), ("test", "^-")]:
        ax.plot(lams, [fine_results[l][sp][met] for l in lams], m, label=sp, ms=5)
    ax.set(xlabel="λ_max", ylabel=ylabel, title=ylabel, xscale="log")
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle("Fine λ Grid Search — Strategy A", fontsize=14)
plt.tight_layout(); plt.savefig("pgnn_fine_lambda.png", dpi=150, bbox_inches="tight"); plt.show()

evaluate(fine_models[best_fine_lam], f"PGNN+λA (fine, λ_max={best_fine_lam})")

## 16 · Save Predictions & Summary

In [ ]:
# ── Save full predictions from best model ─────────────────────────
out_df = data.copy()
out_df["residual"] = out_df["sigma_true"] - out_df["sigma_pred"]

y_full, params_full = predict(best_model, list(range(len(data))))
for pname in ["alpha", "n", "Q", "lnA"]:
    out_df[f"learned_{pname}"] = params_full[pname].flatten()

out_df.to_csv("pgnn_predictions.csv", index=False)
print("Saved pgnn_predictions.csv")

# ── Final summary ────────────────────────────────────────────────
n_params = sum(p.numel() for p in m2.parameters())
print(f"\nArchitecture : PGNN [3 → {' → '.join(map(str, CFG.hidden_dims))} → 4 heads → Arrhenius]")
print(f"Parameters   : {n_params:,}")
print(f"Data         : {len(data):,} pts, {data['condition'].nunique()} conditions")
print(f"Split        : {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")

print(f"\n{'Model':<16} {'Ep':>5} {'Test R²':>9} {'RMSE':>9} {'AARE':>8}")
print("─" * 50)
for tag in ["PGNN", "PGNN+λA", "PGNN+λB", "PGNN+λC"]:
    t = results[tag]["test"]
    print(f"{tag:<16} {histories[tag].best_epoch+1:>5} {t['r2']:>9.6f} {t['rmse']:>9.4f} {t['aare']:>7.2f}%")

print(f"\nBest overall : {best_tag}")
print(f"MC Dropout   : PICP = {picp:.1f}%   MPIW = {mpiw:.2f} MPa")

## 17 · ANN Baseline (Black-Box MLP)

Train a standard MLP with the same architecture but no Arrhenius layer,
for direct comparison with the PGNN variants.

In [ ]:
# ── ANN Model Definition ──────────────────────────────────────
class FlowStressANN(nn.Module):
    """Black-box MLP: [T_K, ln(ε̇), ε] → σ. No physics embedded."""
    def __init__(self, input_dim=3, hidden_dims=None, dropout=CFG.dropout):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = CFG.hidden_dims
        layers = []
        in_d = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# ── ANN Training ─────────────────────────────────────────────
def train_ann(verbose_every=50):
    seed_everything()
    model = FlowStressANN().to(CFG.device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=CFG.scheduler_factor,
        patience=CFG.scheduler_patience, min_lr=1e-6)
    criterion = nn.MSELoss()

    # Build ANN dataloaders (scaled features & targets, no physics inputs)
    def _ann_loader(idx_list, shuffle):
        X_s = scaler_X.transform(X_all[idx_list]).astype(np.float32)
        y_s = scaler_y.transform(y_all[idx_list]).astype(np.float32)
        ds = TensorDataset(torch.tensor(X_s), torch.tensor(y_s))
        return DataLoader(ds, batch_size=CFG.batch_size if shuffle else 256, shuffle=shuffle)

    loader_tr = _ann_loader(train_idx, True)
    loader_va = _ann_loader(val_idx, False)

    best_val, best_state, wait, best_ep = float("inf"), None, 0, 0

    for ep in range(CFG.epochs):
        model.train()
        s_loss, n = 0.0, 0
        for X_b, y_b in loader_tr:
            X_b, y_b = X_b.to(CFG.device), y_b.to(CFG.device)
            loss = criterion(model(X_b), y_b)
            opt.zero_grad(); loss.backward(); opt.step()
            s_loss += loss.item() * len(X_b); n += len(X_b)

        model.eval()
        v_sum = 0.0
        with torch.no_grad():
            for X_b, y_b in loader_va:
                X_b, y_b = X_b.to(CFG.device), y_b.to(CFG.device)
                v_sum += criterion(model(X_b), y_b).item() * len(X_b)
        val_loss = v_sum / len(val_idx)
        sched.step(val_loss)

        if val_loss < best_val:
            best_val, best_ep, wait = val_loss, ep, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if (ep+1) % verbose_every == 0 or ep == 0:
            lr_now = opt.param_groups[0]["lr"]
            print(f"  [ANN] ep {ep+1:4d}  train={s_loss/n:.6f}  val={val_loss:.6f}  lr={lr_now:.1e}")
        if wait >= CFG.patience:
            print(f"  [ANN] Early stop ep {ep+1} (best: {best_ep+1})")
            break

    model.load_state_dict(best_state)
    model.eval()
    print(f"  [ANN] Best val={best_val:.6f} @ ep {best_ep+1}")
    return model


def predict_ann(model, idx_list):
    """Predict σ (MPa) from ANN model."""
    model.eval()
    X_s = torch.tensor(scaler_X.transform(X_all[idx_list]).astype(np.float32), device=CFG.device)
    with torch.no_grad():
        y_s = model(X_s).cpu().numpy()
    return scaler_y.inverse_transform(y_s).flatten()


print("━" * 60, "\nModel 1 — ANN (black-box MLP)\n" + "━" * 60)
ann_model = train_ann()

# Evaluate
y_ann_test = predict_ann(ann_model, test_idx)
ann_metrics = metrics(y_all[test_idx], y_ann_test)
print(f"\n  ANN Test:  R²={ann_metrics['r2']:.6f}  RMSE={ann_metrics['rmse']:.4f}  AARE={ann_metrics['aare']:.2f}%")

torch.save(ann_model.state_dict(), "best_ann.pt")
print("  Saved best_ann.pt")

## 18 · Full Model Comparison (SCAM + ANN + PGNN Variants)

In [ ]:
# ── Generate predictions for all models on full dataset ───────
all_idx = list(range(len(data)))

# SCAM predictions (valid only 250–450°C)
sigma_scam_pred = scam_sigma_np(eps_raw, T_K_raw, sr_raw)
scam_valid = np.isfinite(sigma_scam_pred) & (T_K_raw >= CFG.scam_temp_min_K)
sigma_scam_pred[~scam_valid] = np.nan
sigma_scam_pred = np.clip(sigma_scam_pred, 0, 500)

data["pred_SCAM"] = sigma_scam_pred
data["pred_ANN"]  = predict_ann(ann_model, all_idx)

# PGNN predictions (already have pred_noλ and pred_λA from Section 14)
# Also store predictions from all PGNN variants
for tag in ["PGNN", "PGNN+λA", "PGNN+λB", "PGNN+λC"]:
    safe_col = "pred_" + tag.replace("+", "_").replace("λ", "lam")
    yp, _ = predict(models[tag], all_idx)
    data[safe_col] = yp.flatten()

# Model column mapping for analysis
MODEL_COLS = {
    "SCAM":     "pred_SCAM",
    "ANN":      "pred_ANN",
    "PGNN":     "pred_PGNN",
    "PGNN+λA":  "pred_PGNN_lamA",
    "PGNN+λB":  "pred_PGNN_lamB",
    "PGNN+λC":  "pred_PGNN_lamC",
}

# Store learned params from PGNN and PGNN+λA
_, params_pgnn = predict(models["PGNN"], all_idx)
_, params_la   = predict(models["PGNN+λA"], all_idx)
for pname in ["alpha", "n", "Q", "lnA"]:
    data[f"learned_{pname}_noL"] = params_pgnn[pname].flatten()
    data[f"learned_{pname}_lA"]  = params_la[pname].flatten()

# ── Full comparison table ─────────────────────────────────────
print(f"\n{'Model':<16} {'Train R²':>9} {'Val R²':>9} {'Test R²':>9} {'RMSE':>9} {'AARE':>9}")
print("─" * 60)
for mname, col in MODEL_COLS.items():
    row = []
    for sp, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        yt = y_all[idx].flatten()
        yp = data.loc[idx, col].values
        valid = np.isfinite(yp)
        if valid.sum() < 2:
            row.append({"r2": float("nan"), "rmse": float("nan"), "aare": float("nan")})
        else:
            row.append(metrics(yt[valid].reshape(-1,1), yp[valid].reshape(-1,1)))
    print(f"{mname:<16} {row[0]['r2']:>9.6f} {row[1]['r2']:>9.6f} {row[2]['r2']:>9.6f} "
          f"{row[2]['rmse']:>8.4f} {row[2]['aare']:>8.2f}%")

## 19 · Per-Condition Metrics (R², RMSE, AARE)

Compute R², RMSE, and AARE for every condition × model × split.
This provides the detailed breakdown needed for the paper's results table.

In [ ]:
# ── Compute per-condition metrics ─────────────────────────────
def metrics_full(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    valid = np.isfinite(yp)
    if valid.sum() < 2:
        return {"R2": float("nan"), "RMSE": float("nan"), "AARE": float("nan")}
    yt_v, yp_v = yt[valid], yp[valid]
    return {
        "R2":   float(r2_score(yt_v, yp_v)),
        "RMSE": float(np.sqrt(mean_squared_error(yt_v, yp_v))),
        "AARE": aare(yt_v, yp_v),
    }

rows = []
for cond in data["condition"].unique():
    sub = data[data["condition"] == cond]
    T_C = sub["T_C"].iloc[0]
    sr  = sub["strain_rate"].iloc[0]
    for sp in ["train", "val", "test"]:
        s = sub[sub["split"] == sp]
        if s.empty:
            continue
        yt = s["sigma_true"].values
        for mname, col in MODEL_COLS.items():
            yp = s[col].values
            m = metrics_full(yt, yp)
            rows.append({
                "condition": cond, "T_C": T_C, "strain_rate": sr,
                "split": sp, "n_pts": len(s), "model": mname,
                "R2": m["R2"], "RMSE": m["RMSE"], "AARE": m["AARE"],
            })

cond_df = pd.DataFrame(rows)
cond_df.to_csv("per_condition_metrics.csv", index=False)
print("Saved: per_condition_metrics.csv")

# ── Print test-set table ─────────────────────────────────────
tc = cond_df[cond_df["split"] == "test"].copy()
main_models = ["ANN", "PGNN", "PGNN+λA"]

print(f"\n{'Condition':<18} {'T°C':>5} {'SR':>8}", end="")
for mn in main_models:
    print(f"  {mn+' AARE':>12}", end="")
print()
print("─" * 65)

for cond in sorted(tc["condition"].unique(), key=lambda c: (
        tc[tc["condition"]==c]["T_C"].iloc[0],
        tc[tc["condition"]==c]["strain_rate"].iloc[0])):
    sub = tc[tc["condition"] == cond]
    T_C = sub["T_C"].iloc[0]
    sr  = sub["strain_rate"].iloc[0]
    print(f"{cond:<18} {T_C:5.0f} {sr:8.4f}", end="")
    for mn in main_models:
        row = sub[sub["model"] == mn]
        if row.empty or np.isnan(row["AARE"].iloc[0]):
            print(f"  {'N/A':>12}", end="")
        else:
            print(f"  {row['AARE'].iloc[0]:>11.2f}%", end="")
    print()

# ── Summary CSV ──────────────────────────────────────────────
summary_rows = []
for sp in ["train", "val", "test"]:
    idx_list = {"train": train_idx, "val": val_idx, "test": test_idx}[sp]
    yt = y_all[idx_list].flatten()
    for mname, col in MODEL_COLS.items():
        yp = data.loc[idx_list, col].values
        m = metrics_full(yt, yp)
        summary_rows.append({"split": sp, "model": mname, **m})
pd.DataFrame(summary_rows).to_csv("model_comparison_summary.csv", index=False)
print("\nSaved: model_comparison_summary.csv")

### 19a · Per-Condition Heatmaps (Test R², RMSE, AARE)

In [ ]:
tc = cond_df[cond_df["split"] == "test"].copy()
models_to_plot = ["ANN", "PGNN", "PGNN+λA"]

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
for col_i, mname in enumerate(models_to_plot):
    m_data = tc[tc["model"] == mname]
    for row_i, (met, cmap, vmin, vmax, fmt) in enumerate([
        ("R2",   "RdYlGn",   0.7,  1.0,  ".3f"),
        ("RMSE", "RdYlGn_r", 0,    30,   ".1f"),
        ("AARE", "RdYlGn_r", 0,    25,   ".1f"),
    ]):
        ax = axes[row_i, col_i]
        piv = m_data.pivot_table(index="T_C", columns="strain_rate", values=met)
        if piv.empty:
            ax.set_visible(False); continue
        im = ax.imshow(piv.values, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels([f"{sr:.3f}" for sr in piv.columns], rotation=45, fontsize=8)
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels([f"{T:.0f}" for T in piv.index], fontsize=8)
        for i in range(len(piv.index)):
            for j in range(len(piv.columns)):
                v = piv.values[i, j]
                if np.isfinite(v):
                    color = "white" if (met != "R2" and v > vmax*0.6) or (met == "R2" and v < 0.85) else "black"
                    ax.text(j, i, f"{v:{fmt}}", ha="center", va="center", fontsize=7, color=color)
        plt.colorbar(im, ax=ax, shrink=0.8)
        if col_i == 0: ax.set_ylabel(f"{met}\nTemperature (°C)")
        if row_i == 0: ax.set_title(mname, fontsize=13, fontweight="bold")
        if row_i == 2: ax.set_xlabel("Strain Rate (s⁻¹)")

plt.suptitle("Per-Condition Metrics — Test Set", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("fig_percondition_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

### 19b · Per-Condition AARE Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

conditions_sorted = sorted(tc["condition"].unique(), key=lambda c: (
    tc[(tc["condition"]==c) & (tc["model"]=="PGNN+λA")]["AARE"].values[0]
    if len(tc[(tc["condition"]==c) & (tc["model"]=="PGNN+λA")]) > 0 else 999
), reverse=True)

x = np.arange(len(conditions_sorted))
w = 0.25
colors = {"ANN": "#e41a1c", "PGNN": "#ff7f00", "PGNN+λA": "#377eb8"}

for i, mname in enumerate(["ANN", "PGNN", "PGNN+λA"]):
    vals = []
    for cond in conditions_sorted:
        row = tc[(tc["condition"]==cond) & (tc["model"]==mname)]
        vals.append(row["AARE"].iloc[0] if len(row) > 0 and np.isfinite(row["AARE"].iloc[0]) else 0)
    ax.bar(x + i*w - w, vals, w, label=mname, color=colors[mname], alpha=0.8)

T_C_map = tc.groupby("condition")["T_C"].first()
SR_map  = tc.groupby("condition")["strain_rate"].first()
ax.set_xticks(x)
ax.set_xticklabels([f"{T_C_map[c]:.0f}°C\n{SR_map[c]:.3f}" for c in conditions_sorted],
                   fontsize=7, rotation=45, ha="right")
ax.set_ylabel("AARE (%)")
ax.set_title("Per-Condition Test AARE — Model Comparison (worst → best)")
ax.legend()
ax.axhline(10, color="gray", ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig("fig_percondition_bar.png", dpi=300, bbox_inches="tight")
plt.show()

## 20 · Physical Insight — Learned Arrhenius Parameters

Compare the PGNN+λA learned parameters (α, n, Q, ln A) against
SCAM polynomial values and known physical ranges for Al alloys.

In [ ]:
eps_smooth = np.linspace(0.05, 0.27, 200)
scam_curves = {k: np.polyval(v, eps_smooth) for k, v in SCAM_POLY_COEFFS.items()}

param_info = {
    "alpha": ("α (MPa⁻¹)", 1.0),
    "n":     ("n (stress exponent)", 1.0),
    "Q":     ("Q (kJ/mol)", 1e-3),
    "lnA":   ("ln A", 1.0),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, pname in zip(axes.flat, param_info):
    label, scale = param_info[pname]
    sc = ax.scatter(eps_raw, params_la[pname].flatten() * scale,
                    c=data["T_C"].values, cmap="coolwarm", s=8, alpha=0.5,
                    edgecolors="none", label="PGNN+λA learned")
    # SCAM polynomial (only plot where reasonable)
    scam_y = scam_curves[pname] * scale
    bounds = {"alpha": (0, 0.1), "n": (0, 20), "Q": (50, 400), "lnA": (5, 60)}
    lo, hi = bounds[pname]
    mask_ok = (scam_y > lo) & (scam_y < hi)
    if mask_ok.any():
        ax.plot(eps_smooth[mask_ok], scam_y[mask_ok], "k--", lw=2, alpha=0.7,
                label="SCAM polynomial (250–450°C)")
    ax.set_xlabel("True Strain"); ax.set_ylabel(label)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc="best")
    plt.colorbar(sc, ax=ax, label="T (°C)", shrink=0.8)

plt.suptitle("Learned Arrhenius Parameters vs SCAM — PGNN+λA", fontsize=14)
plt.tight_layout()
plt.savefig("fig_learned_params_vs_scam.png", dpi=300, bbox_inches="tight")
plt.show()

### 20a · Parameter Distributions by Temperature

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
temps_sorted = sorted(data["T_C"].unique())

for ax, pname in zip(axes.flat, param_info):
    label, scale = param_info[pname]
    box_data = [data[data["T_C"] == T][f"learned_{pname}_lA"].values * scale for T in temps_sorted]
    bp = ax.boxplot(box_data, labels=[f"{T:.0f}" for T in temps_sorted],
                    patch_artist=True, widths=0.6, showfliers=False)
    cmap = plt.cm.coolwarm
    for i, patch in enumerate(bp["boxes"]):
        frac = i / (len(temps_sorted) - 1) if len(temps_sorted) > 1 else 0.5
        patch.set_facecolor(cmap(frac)); patch.set_alpha(0.7)
    ax.set_xlabel("Temperature (°C)"); ax.set_ylabel(label)
    ax.set_title(f"{label} vs Temperature", fontsize=12)
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Learned Parameter Distributions by Temperature — PGNN+λA", fontsize=14)
plt.tight_layout()
plt.savefig("fig_param_evolution_T.png", dpi=300, bbox_inches="tight")
plt.show()

### 20b · Activation Energy Analysis

In [ ]:
Q_learned = params_la["Q"].flatten() / 1000  # kJ/mol

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
ax = axes[0]
ax.hist(Q_learned, bins=40, color="steelblue", alpha=0.7, edgecolor="white")
ax.axvline(142, color="red", ls="--", lw=2, label="Al self-diffusion\n(~142 kJ/mol)")
ax.axvspan(130, 160, alpha=0.15, color="red", label="Literature range\nfor Al alloys")
ax.axvline(np.median(Q_learned), color="green", ls="-", lw=2,
           label=f"Median learned\n({np.median(Q_learned):.1f} kJ/mol)")
ax.set_xlabel("Q (kJ/mol)"); ax.set_ylabel("Count")
ax.set_title("Activation Energy Distribution"); ax.legend(fontsize=8)

# Q vs temperature
ax = axes[1]
for sr_val in sorted(data["strain_rate"].unique()):
    mask = data["strain_rate"] == sr_val
    ax.scatter(data.loc[mask, "T_C"], Q_learned[mask], s=10, alpha=0.5, label=f"{sr_val} s⁻¹")
ax.axhline(142, color="red", ls="--", lw=1.5, alpha=0.7)
ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Q (kJ/mol)")
ax.set_title("Learned Q vs Temperature"); ax.legend(fontsize=7)

# Q vs strain
ax = axes[2]
sc = ax.scatter(eps_raw, Q_learned, c=data["T_C"].values, cmap="coolwarm", s=8, alpha=0.5)
scam_Q_smooth = np.polyval(SCAM_POLY_COEFFS["Q"], eps_smooth) / 1000
valid_Q = (scam_Q_smooth > 50) & (scam_Q_smooth < 400)
if valid_Q.any():
    ax.plot(eps_smooth[valid_Q], scam_Q_smooth[valid_Q], "k--", lw=2, label="SCAM polynomial")
ax.axhline(142, color="red", ls="--", lw=1.5, alpha=0.5, label="Al self-diffusion")
ax.set_xlabel("True Strain"); ax.set_ylabel("Q (kJ/mol)")
ax.set_title("Learned Q vs Strain"); ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label="T (°C)")

plt.suptitle("Activation Energy Analysis — PGNN+λA", fontsize=14)
plt.tight_layout()
plt.savefig("fig_activation_energy.png", dpi=300, bbox_inches="tight")
plt.show()

### 20c · Parameter–Feature Correlation Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (tag, suffix) in zip(axes, [("PGNN+λA", "_lA"), ("PGNN (no λ)", "_noL")]):
    param_df = pd.DataFrame({
        "α": data[f"learned_alpha{suffix}"],
        "n": data[f"learned_n{suffix}"],
        "Q (kJ/mol)": data[f"learned_Q{suffix}"] / 1000,
        "ln A": data[f"learned_lnA{suffix}"],
        "T (°C)": data["T_C"],
        "ε̇ (s⁻¹)": data["strain_rate"],
        "ε": data["eps_true"],
    })
    corr = param_df.corr()
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index, fontsize=8)
    for i in range(len(corr)):
        for j in range(len(corr)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(v) > 0.6 else "black")
    ax.set_title(tag, fontsize=12, fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Parameter–Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig("fig_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 21 · Physical Insight Summary

In [ ]:
print("=" * 60)
print("PHYSICAL INSIGHT SUMMARY")
print("=" * 60)

Q_med = np.median(Q_learned)
Q_mean = np.mean(Q_learned)
Q_std = np.std(Q_learned)
print(f"\nActivation Energy Q:")
print(f"  Learned (PGNN+λA):  {Q_mean:.1f} +/- {Q_std:.1f} kJ/mol  (median: {Q_med:.1f})")
print(f"  Al self-diffusion:  ~142 kJ/mol (literature)")
print(f"  Al 6xxx series:     130-180 kJ/mol (typical range)")
if 130 < Q_med < 200:
    print(f"  -> Learned Q is within expected physical range")
else:
    print(f"  -> Learned Q is outside typical range")

alpha_vals = params_la["alpha"].flatten()
print(f"\nStress multiplier alpha:")
print(f"  Learned: {np.mean(alpha_vals):.4f} +/- {np.std(alpha_vals):.4f} MPa-1")
print(f"  Typical for Al alloys: 0.01-0.04 MPa-1")

n_vals = params_la["n"].flatten()
print(f"\nStress exponent n:")
print(f"  Learned: {np.mean(n_vals):.2f} +/- {np.std(n_vals):.2f}")
print(f"  Typical for Al alloys: 3-8")
if np.mean(n_vals) > 5:
    print(f"  n > 5 suggests dislocation climb mechanism")

print(f"\nTemperature dependence of Q:")
for T in sorted(data["T_C"].unique()):
    mask = data["T_C"] == T
    q_at_T = Q_learned[mask]
    print(f"  T={T:3.0f} C: Q = {np.mean(q_at_T):.1f} +/- {np.std(q_at_T):.1f} kJ/mol")

print(f"\nParameter comparison (PGNN vs PGNN+lA, 250-450 C):")
hot_mask = data["T_C"] >= 250
for pname in ["alpha", "n", "Q", "lnA"]:
    v_nolam = data.loc[hot_mask, f"learned_{pname}_noL"]
    v_lam   = data.loc[hot_mask, f"learned_{pname}_lA"]
    scale = 1e-3 if pname == "Q" else 1.0
    unit = " kJ/mol" if pname == "Q" else " MPa-1" if pname == "alpha" else ""
    m1, s1 = v_nolam.mean()*scale, v_nolam.std()*scale
    m2, s2 = v_lam.mean()*scale, v_lam.std()*scale
    print(f"  {pname:<5}: PGNN = {m1:.3f}+/-{s1:.3f}  |  PGNN+lA = {m2:.3f}+/-{s2:.3f}{unit}")

## 22 · Final Comprehensive Summary

In [ ]:
print("=" * 60)
print("COMPREHENSIVE RESULTS SUMMARY")
print("=" * 60)

n_params_pgnn = sum(p.numel() for p in models["PGNN"].parameters())
n_params_ann  = sum(p.numel() for p in ann_model.parameters())

print(f"\nArchitecture:")
print(f"  ANN:  MLP [3 -> {' -> '.join(map(str, CFG.hidden_dims))} -> 1]  ({n_params_ann:,} params)")
print(f"  PGNN: MLP [3 -> {' -> '.join(map(str, CFG.hidden_dims))} -> 4 heads -> Arrhenius]  ({n_params_pgnn:,} params)")
print(f"\nData: {len(data):,} pts, {data['condition'].nunique()} conditions (RT-450 C, 3 strain rates)")
print(f"Split: {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test")

print(f"\n{'Model':<16} {'Test R2':>9} {'RMSE':>9} {'AARE':>9}")
print("-" * 45)
for mname in ["SCAM", "ANN", "PGNN", "PGNN+lA", "PGNN+lB", "PGNN+lC"]:
    col = MODEL_COLS.get(mname.replace("lA","λA").replace("lB","λB").replace("lC","λC"), None)
    if col is None:
        continue
    yt = y_all[test_idx].flatten()
    yp = data.loc[test_idx, col].values
    valid = np.isfinite(yp)
    if valid.sum() < 2:
        continue
    m = metrics(yt[valid].reshape(-1,1), yp[valid].reshape(-1,1))
    print(f"{mname:<16} {m['r2']:>9.6f} {m['rmse']:>8.4f} {m['aare']:>8.2f}%")

# Per-condition summary for best model
tc_best = cond_df[(cond_df["split"]=="test") & (cond_df["model"]=="PGNN+λA")]
r2_above_09 = (tc_best["R2"] > 0.9).sum()
r2_above_07 = (tc_best["R2"] > 0.7).sum()
aare_below_10 = (tc_best["AARE"] < 10).sum()
print(f"\nPGNN+lA per-condition (test):")
print(f"  R2 > 0.9: {r2_above_09}/21 conditions")
print(f"  R2 > 0.7: {r2_above_07}/21 conditions")
print(f"  AARE < 10%: {aare_below_10}/21 conditions")

print(f"\nMC Dropout: PICP = {picp:.1f}%  MPIW = {mpiw:.2f} MPa")
print(f"\nPhysical parameters (PGNN+lA):")
print(f"  Q     = {np.mean(Q_learned):.1f} +/- {np.std(Q_learned):.1f} kJ/mol")
print(f"  alpha = {np.mean(alpha_vals):.4f} +/- {np.std(alpha_vals):.4f} MPa-1")
print(f"  n     = {np.mean(n_vals):.2f} +/- {np.std(n_vals):.2f}")

print(f"\nFine lambda grid best: lambda={best_fine_lam} "
      f"(val R2={fine_results[best_fine_lam]['val']['r2']:.6f})")

print("\n" + "=" * 60)
print("All outputs saved.")
print("=" * 60)